# Proyecto final: clasificación de tipos de personalidad

Paola Tucumán Alarcón · Diplomado de Machine Learning, módulo FML.

# ANALISIS INICIAL DE DATOS

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, BaggingClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix


In [ ]:
# Cargar las respuestas de la encuesta proporcionada en el módulo.
from pathlib import Path
data_path = Path("data/Datos_pr.csv")
if not data_path.exists():
    data_path = Path("../data/Datos_pr.csv")
df = pd.read_csv(data_path, encoding="cp1252")


In [ ]:
# Obtener las preguntas de la encuesta: Extrae y muestra las preguntas incluidas en tu conjunto de datos.
questions = df.columns.tolist()
print("Preguntas en la encuesta:", questions)


In [ ]:
# Traducciones de las columnas
column_translations = {
    "Response Id": "Id de Respuesta",
    "You regularly make new friends.": "Regularmente haces nuevos amigos.",
    "You spend a lot of your free time exploring various random topics that pique your interest": "Pasas mucho de tu tiempo libre explorando varios temas aleatorios que despiertan tu interés",
    "Seeing other people cry can easily make you feel like you want to cry too": "Ver a otras personas llorar puede hacerte sentir que también quieres llorar",
    "You often make a backup plan for a backup plan.": "A menudo haces un plan de respaldo para un plan de respaldo.",
    "You usually stay calm, even under a lot of pressure": "Por lo general te mantienes tranquilo, incluso bajo mucha presión",
    "At social events, you rarely try to introduce yourself to new people and mostly talk to the ones you already know": "En eventos sociales, rara vez intentas presentarte a nuevas personas y principalmente hablas con las que ya conoces",
    "You prefer to completely finish one project before starting another.": "Prefieres terminar completamente un proyecto antes de comenzar otro.",
    "You are very sentimental.": "Eres muy sentimental.",
    "You like to use organizing tools like schedules and lists.": "Te gusta usar herramientas de organización como horarios y listas.",
    "Even a small mistake can cause you to doubt your overall abilities and knowledge.": "Incluso un pequeño error puede hacerte dudar de tus habilidades y conocimientos en general.",
    "You feel comfortable just walking up to someone you find interesting and striking up a conversation.": "Te sientes cómodo acercándote a alguien que encuentras interesante y comenzando una conversación.",
    "You are not too interested in discussing various interpretations and analyses of creative works.": "No te interesa mucho discutir varias interpretaciones y análisis de obras creativas.",
    "You are more inclined to follow your head than your heart.": "Estás más inclinado a seguir tu cabeza que tu corazón.",
    "You usually prefer just doing what you feel like at any given moment instead of planning a particular daily routine.": "Por lo general prefieres hacer lo que te apetece en cualquier momento en lugar de planificar una rutina diaria particular.",
    "You rarely worry about whether you make a good impression on people you meet.": "Rara vez te preocupas por si das una buena impresión a las personas que conoces.",
    "You enjoy participating in group activities.": "Disfrutas participando en actividades grupales.",
    "You like books and movies that make you come up with your own interpretation of the ending.": "Te gustan los libros y películas que te hacen crear tu propia interpretación del final.",
    "Your happiness comes more from helping others accomplish things than your own accomplishments.": "Tu felicidad proviene más de ayudar a otros a lograr cosas que de tus propios logros.",
    "You are interested in so many things that you find it difficult to choose what to try next.": "Te interesan tantas cosas que te resulta difícil elegir qué probar a continuación.",
    "You are prone to worrying that things will take a turn for the worse.": "Tiendes a preocuparte de que las cosas empeoren.",
    "You avoid leadership roles in group settings.": "Evitas roles de liderazgo en entornos grupales.",
    "You are definitely not an artistic type of person.": "Definitivamente no eres una persona artística.",
    "You think the world would be a better place if people relied more on rationality and less on their feelings.": "Piensas que el mundo sería un lugar mejor si las personas se basaran más en la racionalidad y menos en sus sentimientos.",
    "You prefer to do your chores before allowing yourself to relax.": "Prefieres hacer tus tareas antes de permitirte relajarte.",
    "You enjoy watching people argue.": "Disfrutas viendo a la gente discutir.",
    "You tend to avoid drawing attention to yourself.": "Tiendes a evitar llamar la atención sobre ti mismo.",
    "Your mood can change very quickly.": "Tu estado de ánimo puede cambiar muy rápidamente.",
    "You lose patience with people who are not as efficient as you.": "Pierdes la paciencia con personas que no son tan eficientes como tú.",
    "You often end up doing things at the last possible moment.": "A menudo terminas haciendo las cosas en el último momento posible.",
    "You have always been fascinated by the question of what, if anything, happens after death.": "Siempre te ha fascinado la pregunta de qué, si es que hay algo, sucede después de la muerte.",
    "You usually prefer to be around others rather than on your own.": "Por lo general prefieres estar rodeado de otros en lugar de estar solo.",
    "You become bored or lose interest when the discussion gets highly theoretical.": "Te aburres o pierdes interés cuando la discusión se vuelve muy teórica.",
    "You find it easy to empathize with a person whose experiences are very different from yours.": "Te resulta fácil empatizar con una persona cuyas experiencias son muy diferentes a las tuyas.",
    "You usually postpone finalizing decisions for as long as possible.": "Por lo general pospones tomar decisiones finales el mayor tiempo posible.",
    "You rarely second-guess the choices that you have made.": "Rara vez dudas de las elecciones que has hecho.",
    "After a long and exhausting week, a lively social event is just what you need.": "Después de una semana larga y agotadora, un evento social animado es justo lo que necesitas.",
    "You enjoy going to art museums.": "Disfrutas yendo a museos de arte.",
    "You often have a hard time understanding other people’s feelings.": "A menudo te resulta difícil entender los sentimientos de otras personas.",
    "You like to have a to-do list for each day.": "Te gusta tener una lista de tareas para cada día.",
    "You rarely feel insecure.": "Rara vez te sientes inseguro.",
    "You avoid making phone calls.": 'Evitas hacer llamadas telefónicas',
    'You often spend a lot of time trying to understand views that are very different from your own.': 'A menudo pasas mucho tiempo tratando de entender puntos de vista muy diferentes al tuyo.',
    'In your social circle, you are often the one who contacts your friends and initiates activities.': 'En tu circulo social, a menudo eres tu quien contacta a tus amigos e inicia actividades.',
    'If your plans are interrupted, your top priority is to get back on track as soon as possible.': 'Si tus planes se interrumpen, tu principal prioridad es retomar el rumbo lo antes posible.',
    'You are still bothered by mistakes that you made a long time ago.': 'Aún te molestan los errores que cometiste hace mucho tiempo.',
    'You rarely contemplate the reasons for human existence or the meaning of life.': 'Rara vez contemplas las razones de la existencia humana o el significado de la vida.',
    'Your emotions control you more than you control them.': 'Tus emociones te controlan más de lo que tú las controlas.',
    'You take great care not to make people look bad, even when it is completely their fault.': 'Tienes mucho cuidado de no hacer que las personas se vean mal, incluso cuando es completamente su culpa.',
    'Your personal work style is closer to spontaneous bursts of energy than organized and consistent efforts.': 'Tu estilo de trabajo personal es más cercano a ráfagas espontáneas de energía que a esfuerzos organizados y consistentes.',
    'When someone thinks highly of you, you wonder how long it will take them to feel disappointed in you.': 'Cuando alguien piensa muy bien de ti, te preguntas cuánto tiempo les tomará sentirse decepcionados contigo.',
    'You would love a job that requires you to work alone most of the time.': 'Te encantaría un trabajo que requiera que trabajes solo la mayor parte del tiempo.',
    'You believe that pondering abstract philosophical questions is a waste of time.': 'Crees que reflexionar sobre preguntas filosóficas abstractas es una pérdida de tiempo.',
    'You feel more drawn to places with busy, bustling atmospheres than quiet, intimate places.': 'Te sientes más atraído por lugares con atmósferas ocupadas y bulliciosas que por lugares tranquilos e íntimos.',
    'You know at first glance how someone is feeling.': 'Sabes de un vistazo cómo se siente alguien.',
    'You often feel overwhelmed.': 'A menudo te sientes abrumado.',
    'You complete things methodically without skipping over any steps.': 'Terminas las cosas de manera metódica sin omitir ningún paso.',
    'You are very intrigued by things labeled as controversial.': 'Te intrigan mucho las cosas etiquetadas como controversiales.',
    'You would pass along a good opportunity if you thought someone else needed it more.': 'Dejarías pasar una buena oportunidad si pensabas que otra persona la necesitaba más.',
    'You struggle with deadlines.': 'Tienes dificultades con los plazos.',
    'You feel confident that things will work out for you.': 'Te sientes confiado de que las cosas saldrán bien para ti.',
    'Personality': 'Personalidad'}

In [ ]:
# Traduce el nombre de las preguntas: Cambia las columnas del dataframe para que se encuentren en español.
df.rename(columns=column_translations, inplace=True)
df.head()

In [ ]:
# Encontrar el número de registros: Determina y muestra el número total de registros en el dataset.
total_records = df.shape[0]
print(f"Número total de registros: {total_records}")

In [ ]:
# Encontrar el número de personas para cada tipo de personalidad: Agrupa los datos por tipo de personalidad y muestra en una gráfica cuántas personas hay en cada grupo.
plt.figure(figsize=(8,6))
personality_counts = df["Personalidad"].value_counts()
sns.lineplot(x=personality_counts.index, y=personality_counts.values, marker='o')
plt.title("Distribución de Tipos de Personalidad")
plt.xticks(rotation=45)
plt.show()

In [ ]:
# Indicar cada grupo de personalidad definido: Enumera y describe brevemente cada grupo de personalidad encontrado en los datos.
personality_counts = df['Personalidad'].value_counts()
print("Conteo de Personalidades Encontradas:", personality_counts)


##### Descripcion de cada personalidad de la encuesta
###### ESFP (Extrovertido, Sensorial, Emocional, Perceptivo)
###### INFJ (Introvertido, Intuitivo, Emocional, Juzgador)
###### ENFP (Extrovertido, Intuitivo, Emocional, Perceptivo)
###### ENTP (Extrovertido, Intuitivo, Lógico, Perceptivo)
###### ESTJ (Extrovertido, Sensorial, Lógico, Juzgador)
###### ISTJ (Introvertido, Sensorial, Lógico, Juzgador)
###### ISTP (Introvertido, Sensorial, Lógico, Perceptivo)
###### ESTP (Extrovertido, Sensorial, Lógico, Perceptivo)
###### ESFJ (Extrovertido, Sensorial, Emocional, Juzgador)
###### ISFP (Introvertido, Sensorial, Emocional, Perceptivo)
###### ENFJ (Extrovertido, Intuitivo, Emocional, Juzgador)
###### INTJ (Introvertido, Intuitivo, Lógico, Juzgador)
###### INTP (Introvertido, Intuitivo, Lógico, Perceptivo)
###### ISFJ (Introvertido, Sensorial, Emocional, Juzgador)
###### ENTJ (Extrovertido, Intuitivo, Lógico, Juzgador)
###### INFP (Introvertido, Intuitivo, Emocional, Perceptivo)

# MODELO SUPERVISADO


In [ ]:
# Convertir la columna respuesta a etiquetas numéricas: Utiliza pd.factorize() para convertir las respuestas categóricas en etiquetas numéricas.
X = df.drop(columns=["Personalidad"])
y = pd.factorize(df["Personalidad"])[0]


In [ ]:
print(X.shape)
print(y.shape)

In [ ]:
# Dividir en conjuntos de entrenamiento y prueba: Utiliza train_test_split de sklearn para separar los datos en conjuntos de entrenamiento y prueba.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=2025)

## Modelo de Árbol de Clasificación

In [ ]:
# Diseñar un modelo de árbol de clasificación simple: Implementa un árbol de clasificación usando DecisionTreeClassifier de sklearn.
## Optimizar la precisión del modelo: Prueba con 4-5 profundidades diferentes y grafica la precisión de cada configuración.
## Predicciones y observaciones: Con el modelo optimizado, realiza una predicción de los datos y anota tus observaciones.

depths = [5, 10, 15, 20]
accuracies = []

for depth in depths:
    tree_clf = DecisionTreeClassifier(max_depth=depth, random_state=2025)
    tree_clf.fit(X_train, y_train)
    y_pred = tree_clf.predict(X_test)
    accuracies.append(accuracy_score(y_test, y_pred))

plt.plot(depths, accuracies, marker='o')
plt.xlabel("Profundidad del Árbol")
plt.ylabel("Precisión")
plt.title("Precisión vs. Profundidad del Árbol")
plt.show()


## Selección de profundidad

La comparación de profundidades sirve para explorar el equilibrio entre complejidad y rendimiento. Para concluir que un árbol se sobreajusta, conviene comparar métricas de entrenamiento y validación; la exactitud de prueba por sí sola no basta.


In [ ]:
############################ MODELO OPTIMIZADO ##########################
# Entrenamiento final con la mejor profundidad
best_depth = depths[accuracies.index(max(accuracies))]
tree_clf = DecisionTreeClassifier(max_depth=best_depth, random_state=2025)
tree_clf.fit(X_train, y_train)

# Predicciones con el modelo optimizado
y_pred = tree_clf.predict(X_test)

# Reporte de clasificación
print(f"Reporte de clasificación para profundidad {best_depth}:")
print(classification_report(y_test, y_pred))


##### La precisión es baja, con valores entre 0.60 y 0.70. Esto significa que el modelo, aunque predice correctamente la clase con un valor moderado, tiende a hacer una cantidad significativa de falsos positivos.
##### El recall varía entre 0.60 y 0.70, lo que indica que el modelo es menos efectivo para capturar instancias de ciertas clases.
##### El f1-score se encuentra en un rango entre 0.62 y 0.69, lo que es relativamente bajo, indicando que el modelo está teniendo dificultades para equilibrar la precisión y el recall de manera efectiva.
##### La precisión global del modelo es 0.65, lo que significa que el modelo solo está prediciendo correctamente el 65% de las instancias en el conjunto de prueba.












##### El modelo tiene un desempeño moderado, con una precisión promedio de 0.65. Aunque no es pésimo, hay espacio para mejorar, y la técnica de sobreajuste parece ser un factor. Se puede considerar aplicar técnicas de ajuste y validación cruzada para optimizar aún más el modelo.

## Modelo de Bagging de Clasificación

In [ ]:
# Obtener un modelo de bagging de clasificación: Utiliza BaggingClassifier con un DecisionTreeClassifier. Optimiza la profundidad del árbol.
## Graficar precisión vs. profundidad del árbol: Muestra cómo varía la precisión con la profundidad del árbol.
## Predicciones y observaciones: Con el modelo optimizado, realiza una predicción y anota tus observaciones.

depths = [5, 10, 15, 20]
accuracies = []

# Loop para evaluar diferentes profundidades del árbol dentro del Bagging
for depth in depths:
    tree_clf = DecisionTreeClassifier(max_depth=depth, random_state=2025)
    bagging_clf = BaggingClassifier(estimator=tree_clf, n_estimators=50, random_state=2025)
    bagging_clf.fit(X_train, y_train)

    # Predicciones y cálculo de precisión
    y_pred = bagging_clf.predict(X_test)
    accuracies.append(accuracy_score(y_test, y_pred))

# Graficar la precisión vs. profundidad del árbol
plt.plot(depths, accuracies, marker='o')
plt.xlabel("Profundidad del Árbol")
plt.ylabel("Precisión")
plt.title("Precisión vs. Profundidad del Árbol (Bagging con DecisionTree)")
plt.show()

In [ ]:
# Prediccion con el modelo optimizado
optimized_tree_clf = DecisionTreeClassifier(max_depth=15, random_state=2025)
optimized_bagging_clf = BaggingClassifier(estimator=optimized_tree_clf, n_estimators=50, random_state=2025)
optimized_bagging_clf.fit(X_train, y_train)

# Realizar predicciones en el conjunto de prueba
y_pred_optimized = optimized_bagging_clf.predict(X_test)

# Evaluar el desempeño
accuracy_optimized = accuracy_score(y_test, y_pred_optimized)
print(f"Precisión del modelo optimizado: {accuracy_optimized:.2f}")

# Reporte de clasificación
class_report = classification_report(y_test, y_pred_optimized)
print("Reporte de Clasificación:")
print(class_report)

##### Observaciones de Bagging
##### Precisión: La precisión es alta para casi todas las clases, con valores que oscilan entre 0.87 y 0.94. Esto significa que cuando el modelo predice una clase específica, tiene una probabilidad bastante alta de que la predicción sea correcta.
##### Recall: El recall muestra que el modelo está siendo muy eficaz para identificar la mayoría de las instancias de cada clase. Las puntuaciones van desde 0.88 hasta 0.93, lo que significa que el modelo está siendo capaz de capturar un alto porcentaje de las instancias de cada clase.
##### F1-Score: El f1-score, que es una medida combinada de precisión y recall, se encuentra entre 0.88 y 0.93 para la mayoría de las clases. Este puntaje es particularmente útil cuando se desea un equilibrio entre precisión y recall.
##### Accuracy: El accuracy global del modelo es 0.91, lo que significa que el modelo predice correctamente el 91% de las instancias en el conjunto de prueba.

##### Comparacion del arbol ANTES vs DESPUES del Bagging
##### El modelo de árbol de decisión individual a profundidad 15 tiene un rendimiento más bajo, especialmente en términos de Accuracy y recall, lo que sugiere que, aunque es un buen comienzo, no está aprovechando las ventajas de técnicas como el Bagging para reducir la varianza y mejorar el desempeño.
##### En resumen, Bagging mejora significativamente la capacidad de clasificación en comparación con el modelo de árbol de decisión por sí solo, proporcionando mejores resultados en todas las métricas clave.

## Modelo de Random Forest con GridSearchCV

In [ ]:
# Implementar un modelo de Random Forest: Utiliza RandomForestClassifier de sklearn para crear un modelo de clasificación basado en Random Forest.

# Optimización de hiperparámetros con GridSearchCV: Utiliza GridSearchCV para encontrar la mejor combinación de hiperparámetros para el modelo.
## Definir los parámetros a optimizar: Especifica los rangos de los parámetros como n_estimators, max_depth, y min_samples_split.
## Entrenar y validar el modelo: Utiliza cross-validation para entrenar y validar el modelo con diferentes combinaciones de parámetros.
## Seleccionar el mejor modelo: Identifica y guarda el modelo con la mejor precisión.
## Graficar los resultados de GridSearchCV: Muestra los resultados del proceso de optimización para visualizar el rendimiento de las diferentes configuraciones.

# Definir los parámetros a optimizar para GridSearchCV
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 15, 20, 25],
    'min_samples_split': [2, 5, 10],
    'random_state': [2025]}

rf = RandomForestClassifier(random_state=2025)


In [ ]:
# GridSearchCV con 5 veces de validación cruzada
grid_search = GridSearchCV(estimator=rf, param_grid=param_grid, cv=5, n_jobs=-1, verbose=2, scoring='accuracy')

In [ ]:
# Entrenar con GridSearchCV
grid_search.fit(X_train, y_train)

In [ ]:
#Que pasa con los demas parametros?
results = grid_search.cv_results_

mean_test_scores = results['mean_test_score']
params = results['params']

param_combinations = [str(param) for param in params]

plt.figure(figsize=(10,6))
plt.plot(param_combinations, mean_test_scores, marker='o')
plt.xticks(rotation=90)
plt.xlabel('Combinación de Parámetros')
plt.ylabel('Precisión Media')
plt.title('Resultados de GridSearchCV para Random Forest')
plt.show()


In [ ]:
# El mejor modelo encontrado
best_rf_model = grid_search.best_estimator_

# Los mejores parámetros y la mejor precisión
print("Mejores parámetros:", grid_search.best_params_)
print("Mejor precisión:", grid_search.best_score_)

In [ ]:
y_pred_best_rf = best_rf_model.predict(X_test)
print(classification_report(y_test, y_pred_best_rf))

## Interpretación de los modelos

Random Forest obtuvo mejores métricas que el árbol simple y Bagging en el análisis del curso. Una exactitud elevada indica concordancia con las etiquetas de este archivo; no demuestra que los tipos sean válidos ni que el modelo funcione igual con respuestas de otra procedencia.


# ANALISIS Y CONCLUSIONES

## Conclusiones

Los métodos de conjunto mejoraron la clasificación frente al árbol individual en este conjunto de respuestas. Para evaluar un uso fuera del ejercicio haría falta conocer cómo se asignaron las etiquetas y validar con datos independientes de otra fuente.
